In [0]:
# HIGH_GARDEN_CATALOG_PARAMETER
dbutils.widgets.text("catalog", "high_garden")
catalog = dbutils.widgets.get("catalog").strip() or "high_garden"
print(f"Using Unity Catalog: {catalog}")


In [0]:
from pyspark.sql import functions as F

SOURCE_TABLE = f"{catalog}.silver.coffee_consumption"
TARGET_TABLE = f"{catalog}.gold.market_metrics"

df = spark.table(SOURCE_TABLE)

print("Input rows:", df.count())

In [0]:
df = spark.table(SOURCE_TABLE)

display(df.limit(10))

In [0]:
input_rows = df.count()

market_count = (
    df
    .select(
        "country",
        "coffee_type"
    )
    .distinct()
    .count()
)

period_count = (
    df
    .select("crop_year")
    .distinct()
    .count()
)

print("Input rows:", input_rows)
print("Markets:", market_count)
print("Periods:", period_count)

In [0]:
year_bounds = (
    df
    .agg(
        F.min("start_year").alias("earliest_year"),
        F.max("start_year").alias("latest_year")
    )
    .first()
)

earliest_year = year_bounds["earliest_year"]
latest_year = year_bounds["latest_year"]

previous_year = latest_year - 1
recent_start_year = latest_year - 9

print("Earliest year:", earliest_year)
print("Recent start year:", recent_start_year)
print("Previous year:", previous_year)
print("Latest year:", latest_year)

In [0]:
market_metrics = (
    df
    .groupBy(
        "country",
        "coffee_type"
    )
    .agg(
        F.count("*")
        .alias("observations"),

        F.sum("domestic_consumption")
        .alias("historical_consumption"),

        F.avg("domestic_consumption")
        .alias("mean_consumption"),

        F.stddev_samp("domestic_consumption")
        .alias("std_consumption"),

        F.sum("zero_flag")
        .alias("zero_years"),

        F.max(
            F.when(
                F.col("start_year") == earliest_year,
                F.col("domestic_consumption")
            )
        ).alias("consumption_initial"),

        F.max(
            F.when(
                F.col("start_year") == recent_start_year,
                F.col("domestic_consumption")
            )
        ).alias("consumption_recent_start"),

        F.max(
            F.when(
                F.col("start_year") == previous_year,
                F.col("domestic_consumption")
            )
        ).alias("consumption_previous"),

        F.max(
            F.when(
                F.col("start_year") == latest_year,
                F.col("domestic_consumption")
            )
        ).alias("latest_consumption")
    )
)

In [0]:
print("Gold candidate rows:", market_metrics.count())

display(
    market_metrics
    .orderBy(
        F.desc("latest_consumption")
    )
    .limit(10)
)

In [0]:
market_metrics = (
    market_metrics
    .withColumn(
        "volatility_cv",
        F.when(
            F.col("mean_consumption") > 0,
            F.col("std_consumption")
            /
            F.col("mean_consumption")
        )
    )
)

In [0]:
historical_years = (
    latest_year - earliest_year
)

print(
    "Historical CAGR periods:",
    historical_years
)

In [0]:
market_metrics = (
    market_metrics
    .withColumn(
        "cagr_historical",
        F.when(
            (
                F.col("consumption_initial") > 0
            )
            &
            (
                F.col("latest_consumption") > 0
            ),
            F.pow(
                F.col("latest_consumption")
                /
                F.col("consumption_initial"),
                F.lit(
                    1.0 / historical_years
                )
            ) - 1
        )
    )
)

In [0]:
recent_years = (
    latest_year
    - recent_start_year
)

print(
    "Recent CAGR periods:",
    recent_years
)

In [0]:
market_metrics = (
    market_metrics
    .withColumn(
        "cagr_recent",
        F.when(
            (
                F.col(
                    "consumption_recent_start"
                ) > 0
            )
            &
            (
                F.col(
                    "latest_consumption"
                ) > 0
            ),
            F.pow(
                F.col("latest_consumption")
                /
                F.col(
                    "consumption_recent_start"
                ),
                F.lit(
                    1.0 / recent_years
                )
            ) - 1
        )
    )
)

In [0]:
market_metrics = (
    market_metrics
    .withColumn(
        "latest_yoy_growth",
        F.when(
            F.col(
                "consumption_previous"
            ) > 0,
            (
                F.col("latest_consumption")
                -
                F.col(
                    "consumption_previous"
                )
            )
            /
            F.col(
                "consumption_previous"
            )
        )
    )
)

In [0]:
latest_total = (
    df
    .filter(
        F.col("start_year")
        == latest_year
    )
    .agg(
        F.sum(
            "domestic_consumption"
        ).alias("total")
    )
    .first()["total"]
)

print(
    "Latest global consumption:",
    latest_total
)

In [0]:
market_metrics = (
    market_metrics
    .withColumn(
        "latest_market_share",
        F.when(
            F.lit(latest_total) > 0,
            F.col(
                "latest_consumption"
            )
            /
            F.lit(latest_total)
        )
    )
)

In [0]:
print(
    "latest_market_share"
    in market_metrics.columns
)

In [0]:
display(
    market_metrics
    .select(
        "country",
        "latest_consumption",
        "latest_market_share"
    )
    .orderBy(
        F.desc(
            "latest_market_share"
        )
    )
    .limit(10)
)

In [0]:
market_metrics = (
    market_metrics
    .withColumn(
        "zero_percentage",
        F.col("zero_years")
        /
        F.col("observations")
    )
    .withColumn(
        "all_zero_series",
        (
            F.col("zero_years")
            ==
            F.col("observations")
        )
    )
)

In [0]:
market_metrics = market_metrics.select(
    "country",
    "coffee_type",

    "observations",

    "historical_consumption",
    "mean_consumption",
    "std_consumption",
    "volatility_cv",

    "zero_years",
    "zero_percentage",
    "all_zero_series",

    "consumption_initial",
    "consumption_recent_start",
    "consumption_previous",
    "latest_consumption",

    "cagr_historical",
    "cagr_recent",
    "latest_yoy_growth",
    "latest_market_share"
)

In [0]:
required_columns = [
    "country",
    "coffee_type",
    "observations",
    "latest_consumption",
    "volatility_cv",
    "cagr_historical",
    "cagr_recent",
    "latest_yoy_growth",
    "latest_market_share",
    "zero_percentage",
    "all_zero_series"
]

missing_columns = [
    column
    for column in required_columns
    if column not in market_metrics.columns
]

assert not missing_columns, (
    f"Missing Gold columns: "
    f"{missing_columns}"
)

print(
    "Gold schema validation passed."
)

In [0]:
assert (
    market_metrics.count()
    == market_count
), "Unexpected number of Gold markets"

In [0]:
assert (
    market_metrics
    .filter(
        F.col(
            "latest_consumption"
        ) < 0
    )
    .count()
    == 0
), "Negative consumption detected"

In [0]:
assert (
    market_metrics
    .filter(
        F.col(
            "latest_market_share"
        ) < 0
    )
    .count()
    == 0
), "Negative market share detected"

In [0]:
market_share_sum = (
    market_metrics
    .agg(
        F.sum(
            "latest_market_share"
        ).alias("total_share")
    )
    .first()["total_share"]
)

print(
    "Market share sum:",
    market_share_sum
)

In [0]:
assert abs(
    market_share_sum - 1.0
) < 1e-6, (
    "Market shares do not sum to 1"
)

print(
    "All Gold business "
    "validations passed."
)

In [0]:
display(
    market_metrics
    .orderBy(
        F.desc(
            "latest_consumption"
        )
    )
)

In [0]:
(
    market_metrics.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        TARGET_TABLE
    )
)

In [0]:
display(spark.sql(f"""
SELECT *
FROM {catalog}.gold.market_metrics
ORDER BY latest_consumption DESC
LIMIT 15;
"""))


In [0]:
display(spark.sql(f"""
SELECT COUNT(*) AS total_markets
FROM {catalog}.gold.market_metrics;
"""))


In [0]:
display(spark.sql(f"""
SELECT
    country,
    coffee_type,
    latest_consumption,
    ROUND(
        cagr_recent * 100,
        2
    ) AS recent_cagr_pct,
    ROUND(
        latest_yoy_growth * 100,
        2
    ) AS latest_growth_pct,
    ROUND(
        latest_market_share * 100,
        2
    ) AS market_share_pct,
    ROUND(
        volatility_cv,
        3
    ) AS volatility_cv
FROM {catalog}.gold.market_metrics
WHERE cagr_recent IS NOT NULL
ORDER BY latest_consumption DESC;
"""))
